<a href="https://colab.research.google.com/github/BlockNotes-4515/ANIST-Summer-Research-Internship-2K26--/blob/main/3D%20Mapping%20Conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to #3D HeatMap Generation!

# Uploading the Main **Dataset** from the Kaggle

In [3]:
from google.colab import files

uploaded = files.upload()

Saving WeatherMapping.zip to WeatherMapping.zip


# Step-1: Extracting the Zip File **"Weather Mapping"**

In [5]:
import zipfile

with zipfile.ZipFile('WeatherMapping.zip','r') as zip_ref:
    zip_ref.extractall('WeatherData')

print("ZIP extracted successfully")

ZIP extracted successfully


# See files inside **ZIP**

In [6]:
import os

os.listdir('WeatherData')

['Indian_Climate_Dataset_2024_2025.csv']

# Showing you the complete Datset

In [7]:
import pandas as pd

# Construct the full path to the CSV file
csv_file_path = 'WeatherData/Indian_Climate_Dataset_2024_2025.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(csv_file_path)

# Display the complete DataFrame
print(df)

            Date       City           State  Temperature_Max (°C)  \
0     2024-01-01     Mumbai     Maharashtra                  32.5   
1     2024-01-01      Delhi           Delhi                  25.4   
2     2024-01-01  Bengaluru       Karnataka                  37.2   
3     2024-01-01    Chennai      Tamil Nadu                  37.2   
4     2024-01-01    Kolkata     West Bengal                  27.4   
...          ...        ...             ...                   ...   
7305  2025-12-31  Hyderabad       Telangana                  41.8   
7306  2025-12-31  Ahmedabad         Gujarat                  37.7   
7307  2025-12-31     Jaipur       Rajasthan                  44.2   
7308  2025-12-31    Lucknow   Uttar Pradesh                  43.7   
7309  2025-12-31     Bhopal  Madhya Pradesh                  37.8   

      Temperature_Min (°C)  Temperature_Avg (°C)  Humidity (%)  Rainfall (mm)  \
0                     18.0                  25.2          77.6            0.0   
1        

# Step 2: Load dataset

In [9]:
import pandas as pd

data = pd.read_csv(
    "WeatherData/Indian_Climate_Dataset_2024_2025.csv"
)

data.head()

,Date,City,State,Temperature_Max (°C),Temperature_Min (°C),Temperature_Avg (°C),Humidity (%),Rainfall (mm),Wind_Speed (km/h),AQI,AQI_Category,Pressure (hPa),Cloud_Cover (%)
0,2024-01-01,Mumbai,Maharashtra,32.5,18.0,25.2,77.6,0.0,3.3,259,Poor,1020.3,62.1
1,2024-01-01,Delhi,Delhi,25.4,10.7,18.1,84.1,0.0,9.0,130,Moderate,1008.4,46.0
2,2024-01-01,Bengaluru,Karnataka,37.2,30.8,34.0,49.0,3.7,6.6,54,Satisfactory,1008.0,61.3
3,2024-01-01,Chennai,Tamil Nadu,37.2,30.4,33.8,34.2,9.5,9.0,176,Moderate,993.4,70.0
4,2024-01-01,Kolkata,West Bengal,27.4,17.5,22.5,32.2,9.1,9.2,97,Satisfactory,1008.2,56.9


# Step 3: Install coordinate library

In [10]:
!pip install geopy

# Step 4: Convert city names into latitude and longitude

In [11]:
from geopy.geocoders import Nominatim
import pandas as pd

geolocator = Nominatim(user_agent="RADMAP")

# keep only unique cities for faster processing
cities = data[['City']].drop_duplicates()

def get_coordinates(city):
    location = geolocator.geocode(city + ", India")

    if location:
        return pd.Series([
            location.latitude,
            location.longitude
        ])

    return pd.Series([None,None])

cities[['latitude','longitude']] = (
    cities['City'].apply(get_coordinates)
)

data = data.merge(
    cities,
    on='City',
    how='left'
)

data.head()

,Date,City,State,Temperature_Max (°C),Temperature_Min (°C),Temperature_Avg (°C),Humidity (%),Rainfall (mm),Wind_Speed (km/h),AQI,AQI_Category,Pressure (hPa),Cloud_Cover (%),latitude,longitude
0,2024-01-01,Mumbai,Maharashtra,32.5,18.0,25.2,77.6,0.0,3.3,259,Poor,1020.3,62.1,19.054999,72.869203
1,2024-01-01,Delhi,Delhi,25.4,10.7,18.1,84.1,0.0,9.0,130,Moderate,1008.4,46.0,28.613895,77.209006
2,2024-01-01,Bengaluru,Karnataka,37.2,30.8,34.0,49.0,3.7,6.6,54,Satisfactory,1008.0,61.3,12.976794,77.590082
3,2024-01-01,Chennai,Tamil Nadu,37.2,30.4,33.8,34.2,9.5,9.0,176,Moderate,993.4,70.0,13.083694,80.270186
4,2024-01-01,Kolkata,West Bengal,27.4,17.5,22.5,32.2,9.1,9.2,97,Satisfactory,1008.2,56.9,22.572646,88.363895


# Step 5: Clean data

In [12]:
data = data.dropna()

data = data.drop_duplicates()

print("Rows:",len(data))

Rows: 7310


# Step 6: Install heatmap **libraries**

In [13]:
!pip install folium plotly

# **Step 7: Generate 2D Temperature Heatmap**

In [16]:
import folium
from folium.plugins import HeatMap

map_center = [
    data.latitude.mean(),
    data.longitude.mean()
]

m = folium.Map(
    location=map_center,
    zoom_start=5
)

heat_data = data[
[
'latitude',
'longitude',
'Temperature_Avg (°C)'
]
].values.tolist()

HeatMap(heat_data).add_to(m)
m

**Scientific Interpretation**

Blue/Green → lower temperature,

Yellow → moderate,

Red → higher temperature

# **Step 8: Generate 2D Humidity Heatmap**

In [17]:
m2 = folium.Map(
    location=map_center,
    zoom_start=5
)

heat_data2 = data[
[
'latitude',
'longitude',
'Humidity (%)'
]
].values.tolist()

HeatMap(heat_data2).add_to(m2)

m2

Its time to conversion of heatmap 2D-Generation to 3D-Generation

# Step 9: Convert Temperature Heatmap → 3D **Mapping**

In [28]:
import plotly.express as px

fig = px.scatter_3d(
    data,
    x='longitude',
    y='latitude',
    z='Temperature_Avg (°C)',
    color='Temperature_Avg (°C)',
    size='Temperature_Avg (°C)',
    title='3D Temperature Spatial Mapping'
)

fig.show()

**Meaning**
X-axis → Longitude,
Y-axis → Latitude.
Z-axis → Temperature,
Color → Temperature intensity

# Step 10: Convert Humidity Heatmap → 3D **Mapping**

In [ ]:
fig = px.scatter_3d(
    data,
    x='longitude',
    y='latitude',
    z='Humidity (%)',
    color='Humidity (%)',
    size='Humidity (%)',
    title='3D Humidity Spatial Mapping'
)

fig.show()

In [ ]:
import plotly.express as px

# 3D Temperature Spatial Mapping with Night Vision colorscale
fig_temp = px.scatter_3d(
    data,
    x='longitude',
    y='latitude',
    z='Temperature_Avg (°C)',
    color='Temperature_Avg (°C)',
    size='Temperature_Avg (°C)',
    color_continuous_scale='Greens', # Night vision color scale
    title='3D Temperature Spatial Mapping (Night Vision Style)'
)
fig_temp.show()

# 3D Humidity Spatial Mapping with Night Vision colorscale
fig_humidity = px.scatter_3d(
    data,
    x='longitude',
    y='latitude',
    z='Humidity (%)',
    color='Humidity (%)',
    size='Humidity (%)',
    color_continuous_scale='Plasma', # Another night vision-like color scale
    title='3D Humidity Spatial Mapping (Night Vision Style)'
)
fig_humidity.show()

#Prototyping by using the Webcam

In [24]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np
from PIL import Image
import io

def take_photo(filename='photo.jpg', quality=0.8):

    js='''
    async function takePhoto(){

      const div=document.createElement('div');
      const capture=document.createElement('button');

      capture.textContent='Capture';

      div.appendChild(capture);

      document.body.appendChild(div);

      const stream=
      await navigator.mediaDevices.getUserMedia(
      {video:true});

      const video=document.createElement('video');

      document.body.appendChild(video);

      video.srcObject=stream;

      await video.play();

      google.colab.output.setIframeHeight(
      document.body.scrollHeight,true);

      await new Promise(
      resolve=>capture.onclick=resolve
      );

      const canvas=
      document.createElement('canvas');

      canvas.width=video.videoWidth;

      canvas.height=video.videoHeight;

      canvas.getContext('2d')
      .drawImage(
      video,
      0,
      0
      );

      stream.getTracks()
      .forEach(
      track=>track.stop()
      );

      div.remove();

      video.remove();

      return canvas.toDataURL(
      'image/jpeg',
      quality
      );
    }
    '''

    display(Javascript(js))

    data=eval_js('takePhoto()')

    binary=b64decode(
        data.split(',')[1]
    )

    with open(filename,'wb') as f:
        f.write(binary)

    return filename

# Start the Camera **Capture**

In [ ]:
from IPython.display import Image

image_filename = take_photo()

# Display the captured image
Image(filename=image_filename)

<IPython.core.display.Javascript object>

Create the Humidity and the Enviornmental Score **bold text**

In [31]:
data['temp_norm']=(
(data['Temperature_Avg (°C)']
-data['Temperature_Avg (°C)'].min())
/
(data['Temperature_Avg (°C)'].max()
-data['Temperature_Avg (°C)'].min())
)

data['humidity_norm']=(
(data['Humidity (%)']
-data['Humidity (%)'].min())
/
(data['Humidity (%)'].max()
-data['Humidity (%)'].min())
)

data['Environmental_Index']=(
0.5*data['temp_norm']
+
0.5*data['humidity_norm']
)

data.head()

,Date,City,State,Temperature_Max (°C),Temperature_Min (°C),Temperature_Avg (°C),Humidity (%),Rainfall (mm),Wind_Speed (km/h),AQI,AQI_Category,Pressure (hPa),Cloud_Cover (%),latitude,longitude,temp_norm,humidity_norm,Environmental_Index
0,2024-01-01,Mumbai,Maharashtra,32.5,18.0,25.2,77.6,0.0,3.3,259,Poor,1020.3,62.1,19.054999,72.869203,0.307692,0.732308,0.520000
1,2024-01-01,Delhi,Delhi,25.4,10.7,18.1,84.1,0.0,9.0,130,Moderate,1008.4,46.0,28.613895,77.209006,0.020243,0.832308,0.426275
2,2024-01-01,Bengaluru,Karnataka,37.2,30.8,34.0,49.0,3.7,6.6,54,Satisfactory,1008.0,61.3,12.976794,77.590082,0.663968,0.292308,0.478138
3,2024-01-01,Chennai,Tamil Nadu,37.2,30.4,33.8,34.2,9.5,9.0,176,Moderate,993.4,70.0,13.083694,80.270186,0.655870,0.064615,0.360243
4,2024-01-01,Kolkata,West Bengal,27.4,17.5,22.5,32.2,9.1,9.2,97,Satisfactory,1008.2,56.9,22.572646,88.363895,0.198381,0.033846,0.116113


Create movie-style 3D thermal **mapping**

# Create movie-style 3D thermal **mapping**

In [ ]:
!pip install plotly

In [ ]:
import plotly.express as px

fig=px.scatter_3d(

data,

x='longitude',

y='latitude',

z='Environmental_Index',

color='Environmental_Index',

size='Environmental_Index',

color_continuous_scale=[

[0,'green'],
[0.5,'yellow'],
[1,'red']

],

title='Live Thermal Night Vision Mapping'
)

fig.show()

# Generate 2D **HeatMap**

In [ ]:
!pip install folium

In [ ]:
import folium
from folium.plugins import HeatMap

center=[
data.latitude.mean(),
data.longitude.mean()
]

m=folium.Map(
location=center,
zoom_start=5
)

heat_data=data[
[
'latitude',
'longitude',
'Environmental_Index'
]
].values.tolist()

HeatMap(
heat_data
).add_to(m)

m